### Imports

In [1]:
import json
import mlflow
import base64
import getpass
import openai
import os
import pandas as pd
from io import BytesIO
from datasets import load_dataset
from mlflow.models import make_metric
from tqdm import tqdm
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional, List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
import base64
from openai import OpenAI


d:\youtube\experiments\openai-invoice-extraction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000/"
mlflow.openai.autolog()

### Download and load cord v2 dataset

In [3]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

In [4]:
### Load Openai model

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

### Create prompt

In [5]:
schema_dict = {
    "menu":
    {
        "nm": "name of the menu",
        "num": "identification number of menu",
        "unitprice": "unit price of menu",
        "cnt": "quantity of menu",
        "discountprice": "discounted price of menu",
        "price": "total price of menu",
        "itemsubtotal": "price of each menu after discount applied",
        "vatyn": "whether the price includes tax or not",
        "etc": "others",
        "sub": {
            "nm": "name of submenu",
            "unitprice": "unit price of submenu",
            "cnt": "quantity of submenu",
            "price": "total price of submenu",
            "etc": "others"
        }

    },
    "sub_total":
    {
        "price": "subtotal price",
        "discount_price": "discounted price in total",
        "service_price": "service charge",
        "othersvc_price": "added charge other than service charge",
        "tax_price": "tax amount",
        "etc": "others"
    },
    "total":
    {
        "total_price": "total price",
        "etc": "others",
        "cashprice": "amount of price paid in cash",
        "changeprice": "amount of change in cash",
        "creditcardprice": "amount of price paid in credit/debit card",
        "emoneyprice": "amount of price paid in emoney, point",
        "menutype_cnt": "total count of type of menu",
        "menuqty_cnt": "total count of quantity"
    }
}

system_prompt = f"""You are a Vision Language Model specialized in extracting structured data from the invoice receipts.

- Your task is to analyze the provided invoice and extract the relevant information into a well-structured JSON format.
- The invoice receipt includes details such as menu, sub menu, sub total and total.
- Focus on identifying key data fields and ensuring the output adheres to the requested JSON structure.
- Fill the keys only if the information is available in the invoice.

## High-Level Problem Solving Strategy

1. Identify the main sections of the invoice: menu, sub total, and total.
2. For each section, extract the relevant data fields as specified in the schema.
3. Ensure that the output JSON is well-structured and adheres to the provided schema.
4. Do not include None or Null values in the output JSON.
5. Do not add any additional information that is not present in the invoice.

Schema:
{schema_dict}

"""

In [6]:
def pil_to_base64(pil_image):
    buffer = BytesIO()
    pil_image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [16]:
@mlflow.trace
def process_image(image, ground_truth):
    image_base64 = pil_to_base64(image)

    response = client.responses.create(
    model="ft:gpt-4o-2024-08-06:personal:cord-v2-low:Bt81hETw",
    # model="gpt-4.1-nano",
    input=[
        {
                "role": "user",
                "content": [
                    { "type": "input_text", "text": system_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{image_base64}",
                        "detail": "low"
                    },
                ],
            }
        ],
    )
    print(response)
    return response    

In [ ]:
test_dataset = dataset["test"]

# select 5 samples
test_dataset = test_dataset.select(range(5))

for data in tqdm(test_dataset):
    image = data['image']
    ground_truth = json.loads(data['ground_truth'])['gt_parse']

    response = process_image(image, ground_truth)
